# 06 · Agent comparison on configuration D  *(branch `exp/06-agent-comparison`)*

Brief §3.4 (b). **PPO, A2C, DQN, Double DQN, D3QN**, each with its own hyperparameters tuned by Optuna on the
**validation** Sharpe (learning rate, discount, network width, reward scaling + agent-specific: entropy, clip range,
GAE λ, rollout length, minibatch/epochs; batch size, target-update period, exploration, replay ratio).
Then the best configuration of each agent is run with **5 seeds** and tested once.
Tuning trials use seeds 1000+ and are not reported; only the final runs go into `results.csv`.
Checkpoints and trials whose policy **trades fewer than 5 times on validation are not eligible**: in a falling market
"do nothing" or "short once and hold" would otherwise win. Staying in cash is reported separately as the `cash` baseline.
(SAC/TD3 would need a continuous action space — not implemented; the discrete environment is unchanged.)

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/06-agent-comparison"                                  # the branch this notebook belongs to
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "06_agent_comparison")   # exact environment of this run (pip freeze, CUDA, GPU)

In [ ]:
import json, dataclasses, pandas as pd, matplotlib.pyplot as plt, optuna
from rltrade.env import EnvConfig
from rltrade.runner import RunSpec, run_seeds
from rltrade import tune_rl, results as R
optuna.logging.set_verbosity(optuna.logging.WARNING)
AGENTS = ["ppo", "a2c", "dqn", "ddqn", "d3qn"]
SEEDS = [0, 1, 2, 3, 4]
TUNE, N_TRIALS, TUNE_STEPS = True, 15, 500_000
TOTAL_STEPS, EVAL_EVERY = 2_000_000, 100_000
MIN_TRADES = 5                      # a policy must trade at least this often on validation to be eligible
EXP = f"06_agents_mt{MIN_TRADES}"   # the earlier 1-trade-rule results stay under "06_agents"
ENV_D = EnvConfig(state="D")
F = paths.factor_dir(L)

## 1 · Tuning (validation Sharpe)

In [ ]:
best = {}
for agent in AGENTS:
    if TUNE:
        st = tune_rl.tune_agent(agent, L.prepared, F, L.runs, env_cfg=ENV_D, n_trials=N_TRIALS, total_steps=TUNE_STEPS,
                                min_val_trades=MIN_TRADES)
        R.plot_optuna(st, str(L.report / f"rl_optuna_{agent}_mt{MIN_TRADES}"))
        best[agent] = tune_rl.params_to_hp(agent, st.best_params)
        print(agent, "best val Sharpe", round(st.best_value, 3), best[agent])
    else:
        best[agent] = ({}, {})
(L.runs / "tuning" / f"best_hp_mt{MIN_TRADES}.json").write_text(json.dumps({a: {"agent_hp": h, "env": e} for a, (h, e) in best.items()}, indent=2, default=list))

## 2 · Final runs, 5 seeds each

In [ ]:
for agent in AGENTS:
    hp, env_over = best[agent]
    run_seeds(RunSpec(EXP, agent, env=dataclasses.replace(ENV_D, **env_over), agent_hp=hp, min_val_trades=MIN_TRADES,
                      total_steps=TOTAL_STEPS, eval_every=EVAL_EVERY), SEEDS, L.prepared, F, L.runs)
run_seeds(RunSpec(EXP, "buy_and_hold", env=ENV_D), [0], L.prepared, F, L.runs)
run_seeds(RunSpec(EXP, "cash", env=ENV_D), [0], L.prepared, F, L.runs)      # "do nothing" reference

## Results

In [ ]:
df = R.load_results(L.runs); df = df[df.experiment == EXP]
sm = R.summarise(df, metrics=["val_sharpe"] + R.HEADLINE)
table = R.markdown_table(sm, metrics=["val_sharpe"] + R.HEADLINE, group=("agent",))
print(table)
(L.report / f"agents_mt{MIN_TRADES}.md").write_text(f"# Agent comparison, configuration D, eligibility >= {MIN_TRADES} validation trades (test, net of costs)\n\n" + table + "\n", encoding="utf-8")
groups = {a: df[df.agent == a] for a in AGENTS + ["buy_and_hold", "cash"]}
R.plot_curves(df, L.runs, groups, path=L.report / f"agents_mt{MIN_TRADES}_equity.png"); plt.show()
R.plot_curves(df, L.runs, groups, kind="drawdown", path=L.report / f"agents_mt{MIN_TRADES}_drawdown.png"); plt.show()